<a href="https://colab.research.google.com/github/springboardmentor891v/CreditPathAI/blob/Rajath/microsoft_notebooks/microsoft_loan_default.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This is the Real notebook for the dataset from microsoft

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

drive_path = '/content/drive/MyDrive/'

files_to_copy = [
    'Borrower.txt',
    'Loan.txt'

]

colab_path = '/content/'

for file_name in files_to_copy:
    source_path = os.path.join(drive_path, file_name)
    destination_path = os.path.join(colab_path, file_name)
    if os.path.exists(source_path):
        !cp "{source_path}" "{destination_path}"
        print(f"Copied {file_name} to {colab_path}")
    else:
        print(f"File not found in Drive: {file_name}")

print("\nFiles in Colab environment:")
!ls /content/

In [ ]:
import pandas as pd
df_borrower = pd.read_csv("Borrower.txt", sep="\t")
df_loan=pd.read_csv("Loan.txt",sep='\t')

In [ ]:

df = pd.merge(df_borrower, df_loan, on='memberId', how='inner')

print("Merged DataFrame (first 5 rows):")
display(df.head())

print("\nMerged DataFrame Info:")
df.info()

print("\nShape of the merged DataFrame:")
print(df.shape)

In [ ]:
df.drop(['memberId', 'loanId'], axis=1, inplace=True)
print("Columns after dropping 'memberId' and 'loanId':")
display(df.columns)

In [ ]:
df.isnull().sum()

In [ ]:
for col in df.columns:
  print(f" {col}, ----- {df[col].dtype}")

In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = df.drop('loanStatus', axis=1)
y = df['loanStatus']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

In [ ]:
from sklearn.impute import SimpleImputer

numerical_cols_with_nulls = ['numOpenCreditLines', 'isJointApplication', 'loanAmount']
categorical_cols_with_nulls = ['term']

# Numerical imputer
num_imputer = SimpleImputer(strategy='median')
X_train[numerical_cols_with_nulls] = num_imputer.fit_transform(X_train[numerical_cols_with_nulls])
X_test[numerical_cols_with_nulls] = num_imputer.transform(X_test[numerical_cols_with_nulls])

# Categorical imputer
cat_imputer = SimpleImputer(strategy='most_frequent')
X_train[categorical_cols_with_nulls] = cat_imputer.fit_transform(X_train[categorical_cols_with_nulls])
X_test[categorical_cols_with_nulls] = cat_imputer.transform(X_test[categorical_cols_with_nulls])


**ENCODING**

In [ ]:

threshold = 2000

state_counts = X_train['residentialState'].value_counts()

# We create a list of the states that meet our threshold.
top_states = state_counts[state_counts >= threshold].index.tolist()

print(f"Found {len(top_states)} states with counts >= {threshold} in X_train.")
print("These will be kept:", top_states)
print("-" * 30)

print("Applying binning to X_train...")
X_train['residentialState'] = X_train['residentialState'].apply(lambda x: x if x in top_states else 'Other')

print("Applying the same binning rule to X_test...")
X_test['residentialState'] = X_test['residentialState'].apply(lambda x: x if x in top_states else 'Other')



print("\nNew category counts in X_train['residentialState']:")
print(X_train['residentialState'].value_counts())

print("\nNew category counts in X_test['residentialState']:")
print(X_test['residentialState'].value_counts())

In [ ]:
def engineer_date_features(df):
    """Converts 'date' column to datetime and extracts features."""

    df_copy = df.copy()


    df_copy['date'] = pd.to_datetime(df_copy['date'], format='%m/%d/%Y')

    df_copy['loan_issue_year'] = df_copy['date'].dt.year
    df_copy['loan_issue_month'] = df_copy['date'].dt.month
    df_copy['loan_issue_dayofweek'] = df_copy['date'].dt.dayofweek # Monday=0, Sunday=6

    df_copy = df_copy.drop('date', axis=1)

    return df_copy

In [ ]:
#  Apply the function to both X_train and X_test ---
print("Processing date features for X_train...")
X_train = engineer_date_features(X_train)

print("Processing date features for X_test...")
X_test = engineer_date_features(X_test)


print("\nNew date features created. X_train columns:")
print(X_train.columns)
print("\nFirst 5 rows with new features:")
print(X_train[['loan_issue_year', 'loan_issue_month', 'loan_issue_dayofweek']].head())

In [ ]:

# For 'term', we can extract the number directly.
X_train['term'] = X_train['term'].str.replace(' months', '').astype(int)
X_test['term'] = X_test['term'].str.replace(' months', '').astype(int)

employment_map = {
    '< 1 year': 0.5,
    '1 year': 1,
    '2-5 years': 3.5,
    '6-9 years': 7.5,  # Midpoint of 6 and 9
    '10+ years': 10
}
X_train['yearsEmployment'] = X_train['yearsEmployment'].map(employment_map)
X_test['yearsEmployment'] = X_test['yearsEmployment'].map(employment_map)

grade_map = {
    'A1': 0, 'A2': 1, 'A3': 2,
    'B1': 3, 'B2': 4, 'B3': 5,
    'C1': 6, 'C2': 7, 'C3': 8,
    'D1': 9, 'D2': 10, 'D3': 11,
    'E1': 12, 'E2': 13, 'E3': 14
}
X_train['grade'] = X_train['grade'].map(grade_map)
X_test['grade'] = X_test['grade'].map(grade_map)


print("Ordinal encoding complete.")
print("X_train[['term', 'yearsEmployment', 'grade']].head():")
print(X_train[['term', 'yearsEmployment', 'grade']].head())

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# --- Step 1: Identify the nominal columns to encode ---
nominal_cols = ['homeOwnership', 'purpose', 'residentialState']

# --- Step 2: Set up and fit the encoder ---
# handle_unknown='ignore' will prevent errors if a category in test data was not in train data.
# sparse_output=False ensures the output is a standard numpy array.
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# Fit the encoder on the training data and transform it
X_train_ohe_array = ohe.fit_transform(X_train[nominal_cols])

# Transform the test data using the FITTED encoder
X_test_ohe_array = ohe.transform(X_test[nominal_cols])

# Create new DataFrames  ---
ohe_cols = ohe.get_feature_names_out(nominal_cols)

X_train_ohe = pd.DataFrame(X_train_ohe_array, columns=ohe_cols, index=X_train.index)
X_test_ohe = pd.DataFrame(X_test_ohe_array, columns=ohe_cols, index=X_test.index)


# --- Step 4: Combine with the main dataframes and drop the original columns ---
# Drop the original nominal columns
X_train = X_train.drop(columns=nominal_cols)
X_test = X_test.drop(columns=nominal_cols)

# Concatenate the new one-hot encoded columns
X_train = pd.concat([X_train, X_train_ohe], axis=1)
X_test = pd.concat([X_test, X_test_ohe], axis=1)

print("\nOne-Hot Encoding complete.")
print(f"Original shape of X_train: {X_train.shape}")
print("Number of columns added:", len(ohe_cols))

In [ ]:
#encoding the Y , as for XGBoost we can't use strings
target_map = {
    'Current': 0,
    'Default': 1
}

# --- Step 2: Apply the mapping to both y_train and y_test ---
print("Encoding y_train...")
y_train = y_train.map(target_map)

print("Encoding y_test...")
y_test = y_test.map(target_map)

# --- Step 3: Verify the result ---
# It's always good practice to check your work.
print("\nVerification of y_train encoding:")
print(y_train.value_counts())

print("\nVerification of y_test encoding:")
print(y_test.value_counts())

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
numerical_features = [col for col in X_train.columns if X_train[col].nunique() > 2] # A simple way to get non-binary cols

scaler = StandardScaler()

# Fit on training data ONLY
scaler.fit(X_train[numerical_features])

# Transform both train and test data
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

print("Feature scaling complete.")

In [ ]:

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from imblearn.over_sampling import SMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
import lightgbm as lgb
from xgboost import XGBClassifier



counts = y_train.value_counts()

scale_pos_weight = counts[0] / counts[1]

print(f"Target variable encoded. Calculated scale_pos_weight: {scale_pos_weight:.2f}\n")
#applyign smot to avoid imbalance
print("Applying SMOTE to the training data...")
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

models = {
    # --- Models with built-in imbalance handling ---
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(random_state=42, class_weight='balanced'),
    "LightGBM": lgb.LGBMClassifier(random_state=42, class_weight='balanced', verbosity=-1), # verbosity=-1 silences verbose output
    "XGBoost": XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight, use_label_encoder=False, eval_metric='logloss'),

    # --- Models without built-in imbalance handling (we expect lower recall) ---
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
    "Gaussian Naive Bayes": GaussianNB(),
}



for name, model in models.items():
    print(f"--- Training and Evaluating {name} ---")

    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")

    print(classification_report(y_test, y_pred))
    print("-" * 50 + "\n")

In [ ]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_score, recall_score, f1_score

# --- Step 0: Assumptions ---
# This code assumes you have the following ready from our previous steps:
# X_train_resampled, y_train_resampled (Your final, SMOTE-balanced training data)
# X_test_scaled, y_test (Your final, untouched test data)
# All necessary model classes are imported (LogisticRegression, RandomForestClassifier, etc.)

# --- Step 1: Define the Final Models ---
# We use the models without class_weight, as we are training on the SMOTE-balanced data.
models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "LightGBM": lgb.LGBMClassifier(random_state=42, verbosity=-1),
    "XGBoost": XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "AdaBoost": AdaBoostClassifier(random_state=42),
    "K-Nearest Neighbors (KNN)": KNeighborsClassifier(),
    "Gaussian Naive Bayes": GaussianNB(),
}

# This list will store the results for our final summary table.
results_list = []


# --- Step 2: Train Each Model and Evaluate on the Test Set ---
print("--- Starting Final Evaluation on the Unseen Test Data ---")
for name, model in models.items():
    print(f"\n--- Training and Evaluating: {name} ---")

    # Train the model on the full, resampled training data
    model.fit(X_train_resampled, y_train_resampled)

    # Make predictions on the unseen test data
    y_pred = model.predict(X_test_scaled)

    # --- Generate and Print Full Reports ---
    print(f"\nAccuracy: {accuracy_score(y_test, y_pred):.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")

    print(classification_report(y_test, y_pred))
    print("-" * 50 + "\n")